# P0 · 3CAD: training folders and higher resolution (exploratory)

1. **Training folders.** Counts, image sizes, exact duplicates (within train, and train images that also appear
   in test), any non-`good` training folders, and a thumbnail sheet per category to eyeball defects in
   the "defect-free" training data.
2. **560 px instead of 448 px** (40 × 40 patch grid). Tests whether the low 3CAD scores are partly a resolution
   floor. DINOv2-S, codebook vs. coreset, M = K ∈ {64, 256, 1,024}, 3 seeds; the full bank is skipped.

**Runtime (T4):** section 1 about 10 min; section 2 about 6–8 h (feature extraction). Finished categories
are saved to Drive; an interrupted category re-extracts after Run all.

## 1 · Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: DINOv2 extraction will be slow"

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Get the code. Option A: GitHub (default). Option B: set REPO_URL = "" to install
# from bovw-forensics.zip uploaded to MyDrive/bovw-forensics/.
REPO_URL = "https://github.com/M3R2T2L2/bovw_forensics.git"
ZIP_PATH = "/content/drive/MyDrive/bovw-forensics/bovw-forensics.zip"

import os, shutil, subprocess, zipfile
shutil.rmtree("/content/bovw_forensics", ignore_errors=True)
if REPO_URL:
    subprocess.run(["git", "clone", "-q", REPO_URL, "/content/bovw_forensics"], check=True)
else:
    zipfile.ZipFile(ZIP_PATH).extractall("/content/")
    os.rename("/content/bovw-forensics", "/content/bovw_forensics")

%cd /content/bovw_forensics
!pip install -q -e ".[dev]"   # torch, torchvision, transformers come preinstalled on Colab

In [ ]:
# 30-second check that the torch-free core works in this runtime
!python -m pytest -q

In [ ]:
# STL-10 on Drive: restore it here (skips the ~45 min download), or save it after the first download.
import os, shutil
DRIVE_STL = "/content/drive/MyDrive/bovw-forensics/data/stl10_binary"
LOCAL_STL = "/content/data/stl10_binary"
if os.path.isdir(DRIVE_STL) and not os.path.isdir(LOCAL_STL):
    shutil.copytree(DRIVE_STL, LOCAL_STL); print("Restored STL-10 from Drive")
elif os.path.isdir(LOCAL_STL) and not os.path.isdir(DRIVE_STL):
    shutil.copytree(LOCAL_STL, DRIVE_STL); print("Saved STL-10 to Drive for future runtimes")
else:
    print("STL-10 will download on first use; re-run this cell afterwards to save it to Drive"
          if not os.path.isdir(LOCAL_STL) else "STL-10 already local and on Drive")

## 2 · Data

In [ ]:
import torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU: Runtime > Change runtime type > T4 GPU, then Run all.")
import os, glob, shutil, subprocess
from bovw import ad_data

def get_mvtec():
    tar, local = "/content/drive/MyDrive/bovw-forensics/data/mvtec_anomaly_detection.tar.xz", "/content/mvtec"
    if not os.path.isfile(tar):
        raise SystemExit("Put mvtec_anomaly_detection.tar.xz in MyDrive/bovw-forensics/data/ (see notebook 02).")
    if not os.path.isdir(f"{local}/bottle"):
        os.makedirs(local, exist_ok=True)
        subprocess.run(["tar", "-xf", tar, "-C", local], check=True)
    return local

def get_visa():
    tar, local = "/content/drive/MyDrive/bovw-forensics/data/VisA_20220922.tar", "/content/visa"
    if not os.path.isfile(tar) or os.path.getsize(tar) < 1e9:
        ad_data.ensure_archive(tar, url=ad_data.VISA_URL, sha256=None)
    if not os.path.isdir(local):
        os.makedirs(local)
        subprocess.run(["tar", "-xf", tar, "-C", local], check=True)
    return ad_data.find_visa_root(local)

def get_3cad():
    local, archive = "/content/3cad", "/content/3cad_archive"
    if not os.path.isdir(local):
        copy = glob.glob("/content/drive/MyDrive/bovw-forensics/data/3CAD*")
        if not copy:
            raise SystemExit("Put the 3CAD archive in MyDrive/bovw-forensics/data/ (see notebook 06).")
        shutil.copy(copy[0], archive)
        os.makedirs(local)
        magic = open(archive, "rb").read(4)
        fmt = "zip" if magic[:2] == b"PK" else ("gztar" if magic[:2] == b"\x1f\x8b" else "tar")
        shutil.unpack_archive(archive, local, fmt)
        os.remove(archive)
    return ad_data.find_root(local, "Copper_Stator")

ROOT = get_3cad()
print(ROOT)

## 3 · Training-folder inspection

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from bovw import review

RES = "/content/drive/MyDrive/bovw-forensics/results"
rows = []
for cat in ad_data.THREECAD_CATEGORIES:
    info, fig = review.inspect_train_folder(ROOT, cat, n_thumbs=48)
    rows.append(info)
    fig.savefig(f"{RES}/p0_3cad_train_{cat}.png", dpi=110, bbox_inches="tight")
    plt.show()
insp = pd.DataFrame(rows)
insp.to_csv(f"{RES}/p0_3cad_train_inspection.csv", index=False)
insp

## 4 · 560 px run
**Aluminum_New_Ipad** (4,936 test images, about 7.9 million patches at 560 px) needs more RAM than a standard
Colab runtime has: the session crashes and restarts after each run. It is skipped by default; set
`SKIP_CATEGORIES = []` on a high-RAM runtime to include it.

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
from bovw.sweep import load_config
from bovw import anomaly, stats

SKIP_CATEGORIES = ["Aluminum_New_Ipad"]
cfg = load_config("configs/p0_3cad_hires.yaml")
cfg["data"]["root"] = ROOT
cfg["data"]["categories"] = [c for c in ad_data.THREECAD_CATEGORIES if c not in SKIP_CATEGORIES]
hi = anomaly.run(cfg)

## 5 · 560 px vs. 448 px, and codebook vs. coreset at 560 px
Reads the saved results from Drive, so it works in a fresh session (run Setup, then this cell).
Both resolutions are compared on the same categories and seeds only.

In [ ]:
import pandas as pd
from pathlib import Path
from bovw import stats
from bovw.sweep import load_results

RES = "/content/drive/MyDrive/bovw-forensics/results"
hi = load_results(Path(RES) / "p0_3cad_hires.jsonl")
lo = pd.read_csv(f"{RES}/p0_anomaly_3cad.csv")
lo = lo[lo.method.isin(["codebook_dist", "coreset_knn"])]
key = ["category", "method", "k", "vocab_seed"]
for d in (hi, lo):
    d["k"] = d["k"].astype(int)
n_runs = hi.drop_duplicates(key).groupby("category").size()
complete = n_runs[n_runs >= 18].index          # 2 methods x 3 memory sizes x 3 seeds
print("left out (incomplete at 560 px):", sorted(set(n_runs.index) - set(complete)))
common = hi[hi.category.isin(complete)][key].merge(lo[key]).drop_duplicates()
hi_c, lo_c = hi.merge(common), lo.merge(common)
print("categories compared:", sorted(common.category.unique()))
cols = ["image_auroc", "image_auroc_top10", "pixel_auroc", "aupro"]
cmp_ = pd.concat({"448": lo_c, "560": hi_c}, names=["px"]).reset_index(0)
cmp_ = cmp_.groupby(["px", "method", "k", "category"])[cols].mean().groupby(level=[0, 1, 2]).mean().unstack(0).round(3)
cmp_.to_csv(f"{RES}/p0_3cad_hires_vs_448.csv")
ph = stats.compare({"3cad_560": hi_c}, "codebook_dist", "coreset_knn", ks=(64, 256, 1024))
ph.to_csv(f"{RES}/p0_3cad_hires_paired.csv", index=False)
res_gain = stats.compare({"3cad": pd.concat([hi_c.assign(method=hi_c.method + "_560"),
                                              lo_c.assign(method=lo_c.method + "_448")])},
                         "codebook_dist_560", "codebook_dist_448", ks=(64, 256, 1024))
res_gain = pd.concat([res_gain, stats.compare({"3cad": pd.concat([hi_c.assign(method=hi_c.method + "_560"),
                                                                   lo_c.assign(method=lo_c.method + "_448")])},
                                              "coreset_knn_560", "coreset_knn_448", ks=(64, 256, 1024))])
res_gain.to_csv(f"{RES}/p0_3cad_hires_gain.csv", index=False)
display(cmp_)
display(ph.drop(columns=["p_sign", "p_wilcoxon"]).round(3))
res_gain.drop(columns=["p_sign", "p_wilcoxon"]).round(3)